In [13]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning
import xgboost as xgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=InterpolationWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

In [2]:
#  Lokasi proyek 
def cari_root(penanda=("data", "processed")):
    mulai = Path.cwd().resolve()
    for p in (mulai, *mulai.parents):
        if p.joinpath(*penanda).exists():
            return p
    raise FileNotFoundError(f"Folder {'/'.join(penanda)} tidak ditemukan dari {mulai}")


ROOT = cari_root()
DIR_DATA = ROOT / "data" / "processed" / "forecasting"
DIR_TABEL = ROOT / "reports" / "tables" / "forecasting"
DIR_TABEL.mkdir(parents=True, exist_ok=True)

#  Parameter 
HORIZON = 7              # horizon peramalan (hari); tidak boleh melebihi KALENDER_KE_DEPAN notebook 01
UJI_HARI = 28            # blok test: hari terakhir tiap komoditas (kelipatan HORIZON)
N_LIPATAN = 5            # jumlah lipatan validation bergulir (expanding window)
MIN_HARI_LATIH = 365     # data latih minimal pada lipatan pertama
ALPHA = 0.05             # taraf uji ADF dan KPSS
SEED = 42

assert HORIZON >= 1 and N_LIPATAN >= 1
assert UJI_HARI % HORIZON == 0, "UJI_HARI harus kelipatan HORIZON"

#  Pembacaan data tahap 1 
deret = pd.read_csv(DIR_DATA / "deret_waktu_harian.csv", parse_dates=["tanggal"])
kalender = pd.read_csv(DIR_DATA / "kalender_fitur.csv", parse_dates=["tanggal"])
dataset = pd.read_csv(DIR_DATA / "dataset_forecasting.csv", parse_dates=["tanggal"])

#  Validasi kalender dan horizon 
assert (kalender["tanggal"].diff().dropna() == pd.Timedelta(days=1)).all(), "Kalender tidak kontinu"

akhir_kalender = kalender["tanggal"].max()
sisa_hari = (akhir_kalender - deret.groupby("komoditas")["tanggal"].max()).dt.days
assert HORIZON <= sisa_hari.min(), (
    f"HORIZON={HORIZON} melebihi kalender yang tersedia ({int(sisa_hari.min())} hari setelah data terakhir). "
    "Jalankan ulang notebook 01 dengan KALENDER_KE_DEPAN yang lebih besar."
)
assert len(dataset) == len(deret), "dataset_forecasting dan deret_waktu_harian tidak sebaris"

#  Deret per komoditas 
harga_seri, status_seri = {}, {}
for nama, g in deret.sort_values(["komoditas", "tanggal"]).groupby("komoditas"):
    g = g.set_index("tanggal")
    assert (g.index.to_series().diff().dropna() == pd.Timedelta(days=1)).all(), f"{nama}: deret tidak kontinu"
    assert g["harga"].notna().all() and (g["harga"] > 0).all(), f"{nama}: ada harga kosong atau tidak positif"
    harga_seri[nama] = g["harga"].astype(float)
    status_seri[nama] = g["status"]

print(f"Komoditas : {len(harga_seri)}")
print(f"Kalender  : {kalender['tanggal'].min().date()} s.d. {akhir_kalender.date()}")
print(f"HORIZON   : {HORIZON} hari | lolos validasi kalender")
pd.DataFrame({"awal": {k: v.index.min().date() for k, v in harga_seri.items()},
              "akhir": {k: v.index.max().date() for k, v in harga_seri.items()},
              "n_hari": {k: len(v) for k, v in harga_seri.items()}})

Komoditas : 10
Kalender  : 2023-01-01 s.d. 2026-09-07
HORIZON   : 7 hari | lolos validasi kalender


,awal,akhir,n_hari
Bawang Bombay,2023-01-01,2026-08-31,1339
Bayam (Jenis Sayur),2023-01-01,2026-08-31,1339
Buncis,2023-01-01,2026-08-31,1339
Kacang Kedelai (Kualitas Lokal),2023-01-01,2026-08-31,1339
Kangkung (Jenis Panjang),2023-01-01,2026-08-31,1339
Kubis,2023-01-01,2026-08-31,1339
Sawi (Jenis Hijau),2023-01-01,2026-08-31,1339
Terong Ungu (Jenis Sayur),2023-01-01,2026-08-31,1339
Tomat,2023-01-01,2026-08-31,1339
Wortel (Kualitas Lokal),2023-01-01,2026-08-31,1339


In [3]:
# Pembagian data berurutan (skema backtesting: Walk-forward Validation dengan Expanding Window + Final Holdout Test.)
def buat_lipatan(n):
    """Batas posisi untuk deret sepanjang n hari.
    Blok test = UJI_HARI hari terakhir. Lipatan validation berurutan dan berakhir tepat sebelum test.
    Data latih lipatan k = seluruh data sebelum jendela validation lipatan k."""
    uji_awal = n - UJI_HARI
    batas = []
    for k in range(1, N_LIPATAN + 1):
        val_akhir = uji_awal - (N_LIPATAN - k) * HORIZON
        val_awal = val_akhir - HORIZON
        batas.append((k, val_awal, val_akhir))
    return uji_awal, batas


def ambil_lipatan(nama, k):
    """Kembalikan (latih, valid, status_valid) untuk lipatan k (1 s.d. N_LIPATAN)."""
    h, s = harga_seri[nama], status_seri[nama]
    _, batas = buat_lipatan(len(h))
    _, va, vk = batas[k - 1]
    return h.iloc[:va], h.iloc[va:vk], s.iloc[va:vk]


def ambil_uji(nama):
    """Kembalikan (latih_penuh, uji). Dipakai HANYA pada tahap evaluasi akhir (langkah 9)."""
    h = harga_seri[nama]
    n = len(h)
    return h.iloc[: n - UJI_HARI], h.iloc[n - UJI_HARI:]


# ---------- Tabel pembagian ----------
baris_lipatan, baris_uji = [], []
for nama, h in harga_seri.items():
    n = len(h)
    uji_awal, batas = buat_lipatan(n)
    assert batas[0][1] >= MIN_HARI_LATIH, (
        f"{nama}: data latih lipatan pertama hanya {batas[0][1]} hari (minimum {MIN_HARI_LATIH}). "
        "Kurangi N_LIPATAN atau UJI_HARI."
    )
    assert batas[-1][2] == uji_awal, f"{nama}: lipatan terakhir tidak berakhir tepat sebelum test"
    for k, va, vk in batas:
        baris_lipatan.append({
            "komoditas": nama, "lipatan": k, "n_latih": va,
            "latih_akhir": h.index[va - 1].date(),
            "valid_awal": h.index[va].date(), "valid_akhir": h.index[vk - 1].date(),
        })
    baris_uji.append({
        "komoditas": nama, "n_total": n, "latih_valid_akhir": h.index[uji_awal - 1].date(),
        "uji_awal": h.index[uji_awal].date(), "uji_akhir": h.index[-1].date(),
    })

tabel_lipatan = pd.DataFrame(baris_lipatan)
tabel_uji = pd.DataFrame(baris_uji).set_index("komoditas")

# Pemeriksaan: tidak ada tumpang tindih antara data latih, validation, dan test
for nama in harga_seri:
    latih1, _, _ = ambil_lipatan(nama, 1)
    _, uji = ambil_uji(nama)
    assert latih1.index.max() < uji.index.min()

print(f"Blok test: {UJI_HARI} hari terakhir per komoditas (tidak dipakai sampai langkah 9)")
print(f"Validation: {N_LIPATAN} lipatan x {HORIZON} hari, expanding window")
display(tabel_lipatan[tabel_lipatan["komoditas"] == tabel_lipatan["komoditas"].iloc[0]])
tabel_uji

Blok test: 28 hari terakhir per komoditas (tidak dipakai sampai langkah 9)
Validation: 5 lipatan x 7 hari, expanding window


,komoditas,lipatan,n_latih,latih_akhir,valid_awal,valid_akhir
0,Bawang Bombay,1,1276,2026-06-29,2026-06-30,2026-07-06
1,Bawang Bombay,2,1283,2026-07-06,2026-07-07,2026-07-13
2,Bawang Bombay,3,1290,2026-07-13,2026-07-14,2026-07-20
3,Bawang Bombay,4,1297,2026-07-20,2026-07-21,2026-07-27
4,Bawang Bombay,5,1304,2026-07-27,2026-07-28,2026-08-03


,n_total,latih_valid_akhir,uji_awal,uji_akhir
komoditas,,,,
Bawang Bombay,1339,2026-08-03,2026-08-04,2026-08-31
Bayam (Jenis Sayur),1339,2026-08-03,2026-08-04,2026-08-31
Buncis,1339,2026-08-03,2026-08-04,2026-08-31
Kacang Kedelai (Kualitas Lokal),1339,2026-08-03,2026-08-04,2026-08-31
Kangkung (Jenis Panjang),1339,2026-08-03,2026-08-04,2026-08-31
Kubis,1339,2026-08-03,2026-08-04,2026-08-31
Sawi (Jenis Hijau),1339,2026-08-03,2026-08-04,2026-08-31
Terong Ungu (Jenis Sayur),1339,2026-08-03,2026-08-04,2026-08-31
Tomat,1339,2026-08-03,2026-08-04,2026-08-31


### Build Baseline Model

In [4]:
# Baseline: Naive
def prediksi_naive(latih, h=HORIZON):
    return np.repeat(latih.iloc[-1], h)

BASELINE = {"naive": prediksi_naive}

baris = []

for nama in harga_seri:
    for k in range(1, N_LIPATAN + 1):
        latih, valid, st = ambil_lipatan(nama, k)

        for model, fungsi in BASELINE.items():
            pred = fungsi(latih)

            for i in range(HORIZON):
                baris.append({
                    "komoditas": nama,
                    "lipatan": k,
                    "model": model,
                    "h": i + 1,
                    "galat_abs": abs(valid.iloc[i] - pred[i]),
                    "asli": st.iloc[i] == "asli",
                })

galat = pd.DataFrame(baris)

# Hanya menggunakan observasi asli
galat_asli = galat[galat["asli"]]

# Menghitung MAE setiap fold
mae_per_fold = (
    galat_asli
    .groupby(["komoditas", "lipatan"])["galat_abs"]
    .mean()
    .unstack("lipatan")
)

# Menghitung rata-rata MAE dari seluruh fold
mae_per_fold["Rata-rata"] = mae_per_fold.mean(axis=1)

# Membulatkan hasil
mae_per_fold = mae_per_fold.round(1)
mae_per_fold

lipatan,1,2,3,4,5,Rata-rata
komoditas,,,,,,
Bawang Bombay,496.6,286.0,356.9,788.9,312.9,448.2
Bayam (Jenis Sayur),0.0,71.1,904.6,0.0,0.0,195.1
Buncis,3499.6,666.7,3262.3,405.0,642.4,1695.2
Kacang Kedelai (Kualitas Lokal),571.4,428.6,428.6,428.6,428.6,457.1
Kangkung (Jenis Panjang),0.0,71.1,0.0,0.0,0.0,14.2
Kubis,591.6,429.0,547.9,119.1,95.3,356.6
Sawi (Jenis Hijau),321.4,0.0,207.3,147.6,85.7,152.4
Terong Ungu (Jenis Sayur),571.4,57.1,366.7,333.1,195.3,304.7
Tomat,547.9,333.3,404.3,71.4,405.1,352.4


## Model Development

#### Model ARIMA

In [5]:
D_MAKS = 2
def uji_stasioner(z):
    """Stasioner jika ADF menolak H0 (p < ALPHA) DAN KPSS tidak menolak H0 (p > ALPHA)."""
    try:
        pa, pk = adfuller(z, autolag="AIC")[1], kpss(z, nlags="auto")[1]
    except ValueError:                                  # deret konstan
        return np.nan, np.nan, True
    return pa, pk, (pa < ALPHA) and (pk > ALPHA)


def tentukan_d(y):
    """Naikkan d (pembedaan) sampai stasioner, maksimal D_MAKS."""
    z = y.copy()
    for d in range(D_MAKS + 1):
        pa, pk, ok = uji_stasioner(z)
        if d == 0:
            pa0, pk0 = pa, pk                           # p-value pada level (tanpa pembedaan)
        if ok or d == D_MAKS:
            return d, pa0, pk0, ok
        z = z.diff().dropna()


D_UJI, baris = {}, []
for nama in harga_seri:
    for k in range(1, N_LIPATAN + 1):
        latih, _, _ = ambil_lipatan(nama, k)
        d, pa0, pk0, ok = tentukan_d(latih)
        D_UJI[(nama, k)] = d
        baris.append({"komoditas": nama, "lipatan": k, "p_adf": pa0, "p_kpss": pk0,
                      "d": d, "stasioner": ok})

tabel_stasioner = pd.DataFrame(baris)
tabel_stasioner.to_csv(DIR_TABEL / "uji_stasioneritas.csv", index=False)

display(tabel_stasioner.pivot(index="komoditas", columns="lipatan", values="d"))
tabel_stasioner[tabel_stasioner["lipatan"] == 1].set_index("komoditas")[["p_adf", "p_kpss", "d", "stasioner"]].round(4)

lipatan,1,2,3,4,5
komoditas,,,,,
Bawang Bombay,1,1,1,1,1
Bayam (Jenis Sayur),1,1,1,1,1
Buncis,0,0,0,0,0
Kacang Kedelai (Kualitas Lokal),1,1,1,1,1
Kangkung (Jenis Panjang),1,1,1,1,1
Kubis,1,1,1,1,1
Sawi (Jenis Hijau),1,1,1,1,1
Terong Ungu (Jenis Sayur),0,0,0,0,0
Tomat,0,0,0,0,0


,p_adf,p_kpss,d,stasioner
komoditas,,,,
Bawang Bombay,0.0977,0.0100,1,True
Bayam (Jenis Sayur),0.8668,0.0100,1,True
Buncis,0.0002,0.1000,0,True
Kacang Kedelai (Kualitas Lokal),0.0592,0.0100,1,True
Kangkung (Jenis Panjang),0.0116,0.0100,1,True
Kubis,0.0556,0.0522,1,True
Sawi (Jenis Hijau),0.5941,0.0100,1,True
Terong Ungu (Jenis Sayur),0.0130,0.1000,0,True
Tomat,0.0440,0.1000,0,True


In [6]:
# ARIMA: tuning inner (3 split) -> evaluasi outer (lipatan validation)
from itertools import product
from statsmodels.tsa.arima.model import ARIMA

N_INNER, TOL = 3, 0.01
GRID_PQ = list(product(range(4), range(3)))            # p = 0..3, q = 0..2


def ramal_arima(y, order, h=HORIZON):
    trend = "c" if order[1] == 0 else "n"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.asarray(ARIMA(y.asfreq("D"), order=order, trend=trend).fit().forecast(h))


def mae_asli(aktual, pred, status):
    m = (status == "asli").to_numpy()
    return np.mean(np.abs(aktual.to_numpy()[m] - pred[m])) if m.any() else np.nan


def tuning_arima(latih, status, d):
    """Pilih (p,q) dengan MAE inner terkecil; bila selisih <= TOL, ambil yang paling sederhana."""
    awal, skor = len(latih) - N_INNER * HORIZON, {}
    for p, q in GRID_PQ:
        try:
            mae = [mae_asli(latih.iloc[a:a + HORIZON], ramal_arima(latih.iloc[:a], (p, d, q)),
                            status.iloc[a:a + HORIZON]) for a in range(awal, len(latih), HORIZON)]
            if np.isfinite(np.nanmean(mae)):
                skor[(p, q)] = np.nanmean(mae)
        except Exception:
            continue
    batas = min(skor.values()) * (1 + TOL)
    return min((pq for pq, s in skor.items() if s <= batas), key=lambda pq: (sum(pq), skor[pq]))


baris = []
for nama in harga_seri:
    for k in range(1, N_LIPATAN + 1):
        latih, valid, st = ambil_lipatan(nama, k)
        d = D_UJI[(nama, k)]
        p, q = tuning_arima(latih, status_seri[nama].loc[latih.index], d)
        mae = mae_asli(valid, ramal_arima(latih, (p, d, q)), st)
        baris.append({"komoditas": nama, "lipatan": k, "p": p, "d": d, "q": q, "mae": mae})

param_arima = pd.DataFrame(baris)
mae_arima = param_arima.pivot(index="komoditas", columns="lipatan", values="mae")
mae_arima["Rata-rata"] = mae_arima.mean(axis=1)

mae_naive = galat_asli.groupby(["komoditas", "lipatan"])["galat_abs"].mean().groupby("komoditas").mean()
mae_arima["Rasio_vs_Naive"] = mae_arima["Rata-rata"] / mae_naive

param_arima.to_csv(DIR_TABEL / "arima_parameter_per_fold.csv", index=False)
mae_arima.round(3).to_csv(DIR_TABEL / "arima_mae.csv")
display(param_arima.pivot(index="komoditas", columns="lipatan", values=["p", "d", "q"]))
mae_arima.round(3)

p              d              q            
lipatan                          1  2  3  4  5  1  2  3  4  5  1  2  3  4  5
komoditas                                                                   
Bawang Bombay                    2  1  2  0  0  1  1  1  1  1  1  1  1  0  0
Bayam (Jenis Sayur)              0  0  0  0  0  1  1  1  1  1  0  0  0  0  0
Buncis                           1  0  0  0  2  0  0  0  0  0  0  2  2  1  1
Kacang Kedelai (Kualitas Lokal)  0  0  0  0  0  1  1  1  1  1  0  0  0  0  0
Kangkung (Jenis Panjang)         0  0  0  0  0  1  1  1  1  1  0  0  0  0  0
Kubis                            2  2  1  1  0  1  1  1  1  1  1  1  1  1  2
Sawi (Jenis Hijau)               1  1  1  1  0  1  1  1  1  1  1  2  2  2  1
Terong Ungu (Jenis Sayur)        1  1  1  2  3  0  0  0  0  0  2  0  0  1  1
Tomat                            3  1  3  0  3  0  0  0  0  0  2  1  0  2  0
Wortel (Kualitas Lokal)          0  1  1  1  0  1  1  1  1  1  0  1  1  1  0

lipatan,1,2,3,4,5,Rata-rata,Rasio_vs_Naive
komoditas,,,,,,,
Bawang Bombay,520.138,281.659,451.312,788.857,312.857,470.965,1.051
Bayam (Jenis Sayur),0.000,71.143,904.571,0.000,0.000,195.143,1.000
Buncis,3055.198,712.614,2660.538,2224.835,779.410,1886.519,1.113
Kacang Kedelai (Kualitas Lokal),571.429,428.571,428.571,428.571,428.571,457.143,1.000
Kangkung (Jenis Panjang),0.000,71.143,0.000,0.000,0.000,14.229,1.000
Kubis,491.332,463.870,462.806,134.728,206.799,351.907,0.987
Sawi (Jenis Hijau),239.657,32.179,195.324,131.605,86.282,137.009,0.899
Terong Ungu (Jenis Sayur),450.231,186.967,411.737,370.961,219.751,327.929,1.076
Tomat,525.635,324.273,388.381,575.628,457.739,454.331,1.289


#### Model SARIMA

In [7]:
# SARIMA (m = 7): tuning inner (3 split) -> evaluasi outer (lipatan validation)
# Memakai N_INNER, TOL, product, mae_asli, mae_naive dari sel ARIMA dan D_UJI dari sel stasioneritas
from statsmodels.tsa.statespace.sarimax import SARIMAX
from joblib import Parallel, delayed

M = 7
GRID_S = list(product([0, 1], repeat=5))               # (p, q, P, D, Q)


def ramal_sarima(y, order, seasonal, h=HORIZON):
    trend = "c" if order[1] == 0 and seasonal[1] == 0 else "n"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = SARIMAX(y.asfreq("D"), order=order, seasonal_order=(*seasonal, M), trend=trend,
                        enforce_stationarity=False, enforce_invertibility=False)
        return np.asarray(model.fit(disp=False, maxiter=200).forecast(h))


def tuning_sarima(latih, status, d):
    """Pilih (p,q,P,D,Q) dengan MAE inner terkecil; bila selisih <= TOL, ambil yang paling sederhana."""
    awal, skor = len(latih) - N_INNER * HORIZON, {}
    for p, q, P, D, Q in GRID_S:
        try:
            mae = [mae_asli(latih.iloc[a:a + HORIZON],
                            ramal_sarima(latih.iloc[:a], (p, d, q), (P, D, Q)),
                            status.iloc[a:a + HORIZON]) for a in range(awal, len(latih), HORIZON)]
            if np.isfinite(np.nanmean(mae)):
                skor[(p, q, P, D, Q)] = np.nanmean(mae)
        except Exception:
            continue
    batas = min(skor.values()) * (1 + TOL)
    return min((c for c, s in skor.items() if s <= batas), key=lambda c: (sum(c), skor[c]))


def proses(nama, k):
    latih, valid, st = ambil_lipatan(nama, k)
    d = D_UJI[(nama, k)]
    p, q, P, D, Q = tuning_sarima(latih, status_seri[nama].loc[latih.index], d)
    mae = mae_asli(valid, ramal_sarima(latih, (p, d, q), (P, D, Q)), st)
    return {"komoditas": nama, "lipatan": k, "p": p, "d": d, "q": q, "P": P, "D": D, "Q": Q, "m": M, "mae": mae}


baris = Parallel(n_jobs=-2, verbose=5)(
    delayed(proses)(nama, k) for nama in harga_seri for k in range(1, N_LIPATAN + 1))

param_sarima = pd.DataFrame(baris)
mae_sarima = param_sarima.pivot(index="komoditas", columns="lipatan", values="mae")
mae_sarima["Rata-rata"] = mae_sarima.mean(axis=1)
mae_sarima["Rasio_vs_Naive"] = mae_sarima["Rata-rata"] / mae_naive

param_sarima.to_csv(DIR_TABEL / "sarima_parameter_per_fold.csv", index=False)
mae_sarima.round(3).to_csv(DIR_TABEL / "sarima_mae.csv")
display(param_sarima.pivot(index="komoditas", columns="lipatan", values=["p", "d", "q", "P", "D", "Q"]))
mae_sarima.round(3)

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=-2)]: Done  32 out of  50 | elapsed: 17.5min remaining:  9.8min
[Parallel(n_jobs=-2)]: Done  43 out of  50 | elapsed: 19.3min remaining:  3.1min
[Parallel(n_jobs=-2)]: Done  50 out of  50 | elapsed: 19.6min finished


p              d              q              P              D              Q            
lipatan                          1  2  3  4  5  1  2  3  4  5  1  2  3  4  5  1  2  3  4  5  1  2  3  4  5  1  2  3  4  5
komoditas                                                                                                                
Bawang Bombay                    0  1  1  0  0  1  1  1  1  1  0  1  1  0  0  0  0  0  1  0  1  0  1  1  0  1  0  1  1  0
Bayam (Jenis Sayur)              0  0  0  0  0  1  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
Buncis                           0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1  1  0  1  1  1  1  0  0  0  1  1  1  1  1
Kacang Kedelai (Kualitas Lokal)  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  0  0  0  0  1  1  1  1  1  0  0  0  0  0
Kangkung (Jenis Panjang)         0  0  0  0  0  1  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
Kubis                            0  1  1  1  0  1  1  1  1  1  0  1  1  1  1  0  1  0  0  0  0  0  1  1  0  0  0  0  0  0
Sawi (Jenis Hijau)               1  1  1  1  0  1  1  1  1  1  1  1  1  1  1  0  0  0  0  0  0  0  0  0  0  0  1  0  1  0
Terong Ungu (Jenis Sayur)        1  1  1  1  0  0  0  0  0  0  1  0  0  0  1  1  1  0  0  0  1  0  1  1  1  1  1  1  1  1
Tomat                            1  1  1  1  1  0  0  0  0  0  1  1  0  0  0  1  1  0  0  0  0  0  1  1  1  1  0  0  0  0
Wortel (Kualitas Lokal)          0  1  1  1  0  1  1  1  1  1  0  1  1  1  0  0  0  0  0  0  0  0  0  1  0  0  0  0  1  0

lipatan,1,2,3,4,5,Rata-rata,Rasio_vs_Naive
komoditas,,,,,,,
Bawang Bombay,410.694,281.223,503.781,768.086,312.857,455.328,1.016
Bayam (Jenis Sayur),0.000,71.143,904.571,0.000,0.000,195.143,1.000
Buncis,1665.715,998.186,2125.264,886.924,1768.015,1488.821,0.878
Kacang Kedelai (Kualitas Lokal),407.845,90.315,9.493,9.358,9.297,105.262,0.230
Kangkung (Jenis Panjang),0.000,71.143,0.000,0.000,0.000,14.229,1.000
Kubis,591.571,451.225,602.634,306.384,192.455,428.854,1.203
Sawi (Jenis Hijau),239.401,38.635,192.028,122.933,86.289,135.857,0.891
Terong Ungu (Jenis Sayur),439.454,245.186,405.795,400.117,465.525,391.215,1.284
Tomat,530.827,312.267,306.721,209.466,506.703,373.197,1.059


#### Model XGBoost

In [14]:
# XGBoost: model langsung (direct) per komoditas; target = selisih harga hari ke-h terhadap harga terakhir
# Memakai N_INNER, TOL, product, mae_asli, mae_naive dari sel ARIMA

GRID_X = [dict(max_depth=d, n_estimators=n) for d, n in product([2, 3, 5], [100, 300])]
LAGS = [1, 7, 14, 28]
kal = kalender.set_index("tanggal").select_dtypes(include=["number", "bool"]).astype(float)
print("Fitur kalender:", list(kal.columns))

def buat_data(y):
    """Satu baris per (tanggal asal t, h). Fitur hanya memakai data <= t dan kalender tanggal t+h."""
    F = pd.DataFrame({f"sel_{k}": y - y.shift(k) for k in LAGS})
    F["dev_m7"], F["dev_m28"] = y - y.rolling(7).mean(), y - y.rolling(28).mean()
    F["std_7"] = y.rolling(7).std()
    bagian = []
    for h in range(1, HORIZON + 1):
        d = F.assign(h=h, tgl_target=F.index + pd.Timedelta(days=h), target=y.shift(-h) - y)
        bagian.append(d.join(kal, on="tgl_target"))
    return pd.concat(bagian)

def ramal_xgb(D, y, akhir, params):
    """Latih pada baris yang tanggal targetnya <= akhir, lalu ramal HORIZON hari setelah akhir."""
    fitur = [c for c in D.columns if c not in ("tgl_target", "target")]
    latih = D[D["tgl_target"] <= akhir].dropna()
    opsi = {"objective": "reg:absoluteerror", "eta": 0.05, "subsample": 0.8, "seed": SEED,
            "max_depth": params["max_depth"]}
    model = xgb.train(opsi, xgb.DMatrix(latih[fitur], label=latih["target"]),
                      num_boost_round=params["n_estimators"])
    asal = D.loc[akhir].sort_values("h")
    return model.predict(xgb.DMatrix(asal[fitur])) + y.loc[akhir]

def tuning_xgb(D, y, status, akhir_latih):
    """Pilih parameter dengan MAE inner terkecil; bila selisih <= TOL, ambil yang paling sederhana."""
    n, skor = y.index.get_loc(akhir_latih) + 1, {}
    for j, params in enumerate(GRID_X):
        mae = [mae_asli(y.iloc[a:a + HORIZON], ramal_xgb(D, y, y.index[a - 1], params), status.iloc[a:a + HORIZON])
               for a in range(n - N_INNER * HORIZON, n, HORIZON)]
        if np.isfinite(np.nanmean(mae)):
            skor[j] = np.nanmean(mae)
    batas = min(skor.values()) * (1 + TOL)
    j = min((j for j, s in skor.items() if s <= batas),
            key=lambda j: (GRID_X[j]["max_depth"], GRID_X[j]["n_estimators"], skor[j]))
    return GRID_X[j]

baris = []
for nama, y in harga_seri.items():
    D, status = buat_data(y), status_seri[nama]
    for k in range(1, N_LIPATAN + 1):
        latih, valid, st = ambil_lipatan(nama, k)
        akhir = latih.index[-1]
        params = tuning_xgb(D, y, status, akhir)
        baris.append({"komoditas": nama, "lipatan": k, **params,
                      "mae": mae_asli(valid, ramal_xgb(D, y, akhir, params), st)})

param_xgb = pd.DataFrame(baris)
mae_xgb = param_xgb.pivot(index="komoditas", columns="lipatan", values="mae")
mae_xgb["Rata-rata"] = mae_xgb.mean(axis=1)
mae_xgb["Rasio_vs_Naive"] = mae_xgb["Rata-rata"] / mae_naive

param_xgb.to_csv(DIR_TABEL / "xgboost_parameter_per_fold.csv", index=False)
mae_xgb.round(3).to_csv(DIR_TABEL / "xgboost_mae.csv")
display(param_xgb.pivot(index="komoditas", columns="lipatan", values=["max_depth", "n_estimators"]))
mae_xgb.round(3)

Fitur kalender: ['tahun', 'kuartal', 'bulan', 'minggu_tahun', 'hari_dalam_bulan', 'hari_dalam_minggu', 'akhir_pekan', 'libur', 'hari_ke_libur_berikut', 'hari_sejak_libur']


max_depth             n_estimators                    
lipatan                                 1  2  3  4  5            1    2    3    4    5
komoditas                                                                             
Bawang Bombay                           5  3  3  3  2          300  300  300  100  300
Bayam (Jenis Sayur)                     3  2  2  2  2          300  100  100  100  100
Buncis                                  3  5  5  5  5          300  300  300  300  100
Kacang Kedelai (Kualitas Lokal)         2  2  5  5  5          100  100  300  300  300
Kangkung (Jenis Panjang)                2  2  2  2  2          100  100  100  100  100
Kubis                                   2  2  2  2  2          100  100  100  100  100
Sawi (Jenis Hijau)                      3  3  3  2  2          300  300  300  300  300
Terong Ungu (Jenis Sayur)               2  5  5  5  2          100  300  300  300  100
Tomat                                   2  2  2  5  5          100  100  300  300  300
Wortel (Kualitas Lokal)                 5  5  5  5  2          300  300  300  300  100

lipatan,1,2,3,4,5,Rata-rata,Rasio_vs_Naive
komoditas,,,,,,,
Bawang Bombay,477.424,305.905,386.007,788.209,312.443,453.998,1.013
Bayam (Jenis Sayur),0.000,71.143,904.571,165.893,5.768,229.475,1.176
Buncis,3090.542,543.166,2923.186,676.484,858.388,1618.353,0.955
Kacang Kedelai (Kualitas Lokal),571.429,428.571,361.750,245.476,276.656,376.776,0.824
Kangkung (Jenis Panjang),0.000,71.143,0.000,0.000,0.000,14.229,1.000
Kubis,588.228,435.802,542.090,119.407,100.776,357.260,1.002
Sawi (Jenis Hijau),321.429,3.996,100.441,147.571,85.714,131.830,0.865
Terong Ungu (Jenis Sayur),571.429,83.344,360.237,380.811,198.143,318.793,1.046
Tomat,548.147,329.778,316.761,216.370,402.663,362.744,1.029


In [15]:
# Perbandingan MAE rata-rata tiap model (lipatan validation) terhadap baseline naive
tabel_banding = pd.DataFrame({
    "naive":   mae_naive,
    "arima":   mae_arima["Rata-rata"],
    "sarima":  mae_sarima["Rata-rata"],
    "xgboost": mae_xgb["Rata-rata"],
})

kandidat = tabel_banding[["arima", "sarima", "xgboost"]]
menang = kandidat.min(axis=1) < tabel_banding["naive"] - 1e-9      # harus benar-benar lebih kecil dari naive
tabel_banding["pemenang"] = np.where(menang, kandidat.idxmin(axis=1), "naive")
tabel_banding["rasio_vs_naive"] = tabel_banding[["naive", "arima", "sarima", "xgboost"]].min(axis=1) / tabel_banding["naive"]

PEMENANG = tabel_banding["pemenang"].to_dict()
tabel_banding.round(3).to_csv(DIR_TABEL / "perbandingan_model.csv")
print(tabel_banding["pemenang"].value_counts().to_string())
tabel_banding.round(3)

pemenang
naive      5
sarima     2
xgboost    2
arima      1


,naive,arima,sarima,xgboost,pemenang,rasio_vs_naive
komoditas,,,,,,
Bawang Bombay,448.229,470.965,455.328,453.998,naive,1.000
Bayam (Jenis Sayur),195.143,195.143,195.143,229.475,naive,1.000
Buncis,1695.200,1886.519,1488.821,1618.353,sarima,0.878
Kacang Kedelai (Kualitas Lokal),457.143,457.143,105.262,376.776,sarima,0.230
Kangkung (Jenis Panjang),14.229,14.229,14.229,14.229,naive,1.000
Kubis,356.571,351.907,428.854,357.260,arima,0.987
Sawi (Jenis Hijau),152.400,137.009,135.857,131.830,xgboost,0.865
Terong Ungu (Jenis Sayur),304.743,327.929,391.215,318.793,naive,1.000
Tomat,352.400,454.331,373.197,362.744,naive,1.000


In [16]:
# final tuning, hanya model pemenang, pada data pengembangan (semua data kecuali 28 hari test)
PARAM_FINAL, baris = {}, []
for nama, model in PEMENANG.items():
    if model == "naive":
        PARAM_FINAL[nama] = {"model": "naive"}
        continue
    y, st = harga_seri[nama], status_seri[nama]
    latih = y.iloc[:len(y) - UJI_HARI]                      #    1.311 hari
    d, *_ = tentukan_d(latih)

    if model == "arima":
        p, q = tuning_arima(latih, st.loc[latih.index], d)
        par = {"order": (p, d, q)}
    elif model == "sarima":
        p, q, P, D, Q = tuning_sarima(latih, st.loc[latih.index], d)
        par = {"order": (p, d, q), "seasonal": (P, D, Q)}
    else:
        par = {"params": tuning_xgb(buat_data(y), y, st, latih.index[-1])}

    PARAM_FINAL[nama] = {"model": model, "n_latih": len(latih), **par}

tabel_final = pd.DataFrame(PARAM_FINAL).T
tabel_final.to_csv(DIR_TABEL / "parameter_final.csv")
tabel_final

,model,n_latih,order,seasonal,params
Bawang Bombay,naive,NaN,NaN,NaN,NaN
Bayam (Jenis Sayur),naive,NaN,NaN,NaN,NaN
Buncis,sarima,1311,"(0, 0, 1)","(0, 1, 1)",NaN
Kacang Kedelai (Kualitas Lokal),sarima,1311,"(0, 1, 0)","(0, 1, 0)",NaN
Kangkung (Jenis Panjang),naive,NaN,NaN,NaN,NaN
Kubis,arima,1311,"(0, 1, 0)",NaN,NaN
Sawi (Jenis Hijau),xgboost,1311,NaN,NaN,"{'max_depth': 2, 'n_estimators': 300}"
Terong Ungu (Jenis Sayur),naive,NaN,NaN,NaN,NaN
Tomat,naive,NaN,NaN,NaN,NaN
Wortel (Kualitas Lokal),xgboost,1311,NaN,NaN,"{'max_depth': 2, 'n_estimators': 100}"


### Final Tuning

In [17]:
# evaluasi test, parameter terkunci, 4 jendela x 7 hari; tiap jendela diramal dari data sebelum jendela itu
baris = []
for nama, cfg in PARAM_FINAL.items():
    y, st, m = harga_seri[nama], status_seri[nama], cfg["model"]
    awal = len(y) - UJI_HARI
    D = buat_data(y) if m == "xgboost" else None
    for a in range(awal, len(y), HORIZON):
        latih, aktual, s = y.iloc[:a], y.iloc[a:a + HORIZON], st.iloc[a:a + HORIZON]
        if m == "naive":    pred = prediksi_naive(latih)
        elif m == "arima":  pred = ramal_arima(latih, cfg["order"])
        elif m == "sarima": pred = ramal_sarima(latih, cfg["order"], cfg["seasonal"])
        else:               pred = ramal_xgb(D, y, y.index[a - 1], cfg["params"])
        baris.append({"komoditas": nama, "model": m, "jendela": (a - awal) // HORIZON + 1,
                      "mae": mae_asli(aktual, pred, s),
                      "mae_naive": mae_asli(aktual, prediksi_naive(latih), s)})

hasil_test = pd.DataFrame(baris)
tabel_test = hasil_test.groupby("komoditas").agg(model=("model", "first"), mae=("mae", "mean"),
                                                 mae_naive=("mae_naive", "mean"))
tabel_test["rasio_vs_naive"] = tabel_test["mae"] / tabel_test["mae_naive"]

hasil_test.to_csv(DIR_TABEL / "evaluasi_test_per_jendela.csv", index=False)
tabel_test.round(3).to_csv(DIR_TABEL / "evaluasi_test.csv")
tabel_test.round(3)

,model,mae,mae_naive,rasio_vs_naive
komoditas,,,,
Bawang Bombay,naive,606.143,606.143,1.000
Bayam (Jenis Sayur),naive,56.536,56.536,1.000
Buncis,sarima,764.589,505.857,1.511
Kacang Kedelai (Kualitas Lokal),sarima,1017.857,696.429,1.462
Kangkung (Jenis Panjang),naive,0.000,0.000,NaN
Kubis,arima,357.036,357.036,1.000
Sawi (Jenis Hijau),xgboost,108.929,108.929,1.000
Terong Ungu (Jenis Sayur),naive,273.821,273.821,1.000
Tomat,naive,416.857,416.857,1.000


### diagnoostik kubis dan sawi 

In [ ]:
for nama in ["Kubis", "Sawi (Jenis Hijau)"]:
    cfg, y = PARAM_FINAL[nama], harga_seri[nama]
    a = len(y) - UJI_HARI
    pred = (ramal_arima(y.iloc[:a], cfg["order"]) if cfg["model"] == "arima"
            else ramal_xgb(buat_data(y), y, y.index[a - 1], cfg["params"]))
    print(nama, {k: v for k, v in cfg.items() if k != "n_latih"})
    print("  prediksi :", np.round(pred, 1))
    print("  naive    :", np.round(prediksi_naive(y.iloc[:a]), 1))

Kubis {'model': 'arima', 'order': (0, 1, 0)}
  prediksi : [9833. 9833. 9833. 9833. 9833. 9833. 9833.]
  naive    : [9833. 9833. 9833. 9833. 9833. 9833. 9833.]
Sawi (Jenis Hijau) {'model': 'xgboost', 'params': {'max_depth': 2, 'n_estimators': 300}}
  prediksi : [2500. 2500. 2500. 2500. 2500. 2500. 2500.]
  naive    : [2500. 2500. 2500. 2500. 2500. 2500. 2500.]


In [19]:
# Ringkasan akhir: status tiap komoditas pada evaluasi test
t = tabel_test.copy()
setara = np.isclose(t["mae"], t["mae_naive"])
t["status"] = np.select(
    [t["model"] == "naive", setara, t["mae"] > t["mae_naive"]],
    ["naive (menang di validation)", "setara naive", "kalah dari naive di test"],
    default="mengalahkan naive di test")
t.round(3).to_csv(DIR_TABEL / "ringkasan_akhir.csv")
print(t["status"].value_counts().to_string())
t.round(3)

status
naive (menang di validation)    5
kalah dari naive di test        3
setara naive                    2


,model,mae,mae_naive,rasio_vs_naive,status
komoditas,,,,,
Bawang Bombay,naive,606.143,606.143,1.000,naive (menang di validation)
Bayam (Jenis Sayur),naive,56.536,56.536,1.000,naive (menang di validation)
Buncis,sarima,764.589,505.857,1.511,kalah dari naive di test
Kacang Kedelai (Kualitas Lokal),sarima,1017.857,696.429,1.462,kalah dari naive di test
Kangkung (Jenis Panjang),naive,0.000,0.000,NaN,naive (menang di validation)
Kubis,arima,357.036,357.036,1.000,setara naive
Sawi (Jenis Hijau),xgboost,108.929,108.929,1.000,setara naive
Terong Ungu (Jenis Sayur),naive,273.821,273.821,1.000,naive (menang di validation)
Tomat,naive,416.857,416.857,1.000,naive (menang di validation)


In [23]:
for nama in ["Buncis", "Kacang Kedelai (Kualitas Lokal)"]:
    print(f"\n{nama}")
    print("Validation per lipatan:")
    print(pd.DataFrame({"sarima": mae_sarima.loc[nama, range(1, 6)],
                        "naive": mae_per_fold.loc[nama, range(1, 6)]}).T.round(1))
    print("Test per jendela:")
    print(hasil_test[hasil_test["komoditas"] == nama].set_index("jendela")[["mae", "mae_naive"]].T.round(1))
    print("Parameter SARIMA per lipatan:")
    print(param_sarima[param_sarima["komoditas"] == nama].set_index("lipatan")[["p", "q", "P", "D", "Q"]].T)


Buncis
Validation per lipatan:
lipatan       1      2       3      4       5
sarima   1665.7  998.2  2125.3  886.9  1768.0
naive    3499.6  666.7  3262.3  405.0   642.4
Test per jendela:
jendela        1       2       3      4
mae        306.4  1130.3  1306.9  314.8
mae_naive  404.4   881.0   309.4  428.6
Parameter SARIMA per lipatan:
lipatan  1  2  3  4  5
p        0  0  0  0  0
q        1  1  1  1  1
P        1  1  0  1  1
D        1  1  0  0  0
Q        1  1  1  1  1

Kacang Kedelai (Kualitas Lokal)
Validation per lipatan:
lipatan      1      2      3      4      5
sarima   407.8   90.3    9.5    9.4    9.3
naive    571.4  428.6  428.6  428.6  428.6
Test per jendela:
jendela        1      2       3       4
mae          0.0  428.6  1714.3  1928.6
mae_naive  428.6  428.6  1285.7   642.9
Parameter SARIMA per lipatan:
lipatan  1  2  3  4  5
p        1  1  1  1  1
q        1  1  1  1  1
P        1  0  0  0  0
D        1  1  1  1  1
Q        0  0  0  0  0


In [24]:
y = harga_seri["Kacang Kedelai (Kualitas Lokal)"].iloc[-70:]
pd.DataFrame({"minggu": np.arange(len(y)) // 7, "hari": y.index.dayofweek,
              "harga": y.values}).pivot(index="minggu", columns="hari", values="harga")

C:\Users\Eris Alfionita\AppData\Local\Temp\ipykernel_29256\4216984215.py:2: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  pd.DataFrame({"minggu": np.arange(len(y)) // 7, "hari": y.index.dayofweek,


hari,0,1,2,3,4,5,6
minggu,,,,,,,
0,14750.0,14250.0,14250.0,14250.0,14250.0,15750.0,14750.0
1,14250.0,14750.0,14250.0,14250.0,14250.0,15750.0,15750.0
2,14250.0,14250.0,14250.0,14250.0,14250.0,15750.0,15750.0
3,14250.0,14250.0,14250.0,14250.0,14250.0,15750.0,15750.0
4,14250.0,14250.0,14250.0,14250.0,14250.0,15750.0,15750.0
5,14250.0,14250.0,14250.0,14250.0,14250.0,15750.0,15750.0
6,14250.0,14250.0,14250.0,14250.0,14250.0,15750.0,15750.0
7,15750.0,14250.0,14250.0,14250.0,14250.0,15750.0,14250.0
8,14250.0,14250.0,14250.0,14250.0,14250.0,15750.0,14250.0


In [25]:
nama = "Kacang Kedelai (Kualitas Lokal)"
cfg, y = PARAM_FINAL[nama], harga_seri[nama]
awal = len(y) - UJI_HARI
for j in (2, 3):
    a = awal + j * HORIZON
    pred = ramal_sarima(y.iloc[:a], cfg["order"], cfg["seasonal"])
    print(f"Jendela {j + 1}")
    print(pd.DataFrame({"aktual": y.iloc[a:a + HORIZON].values, "sarima": pred.round(0)},
                       index=y.index[a:a + HORIZON].day_name()).T)

Jendela 3
tanggal  Tuesday  Wednesday  Thursday   Friday  Saturday   Sunday   Monday
aktual   14250.0    14250.0   14250.0  14250.0   15750.0  14250.0  14250.0
sarima   15750.0    15750.0   15750.0  15750.0   17250.0  15750.0  17250.0
Jendela 4
tanggal  Tuesday  Wednesday  Thursday   Friday  Saturday   Sunday   Monday
aktual   15750.0    14250.0   14250.0  14250.0   15750.0  15750.0  14250.0
sarima   12750.0    12750.0   12750.0  12750.0   14250.0  12750.0  12750.0
